# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One content page for one client, aggregated over the March 2026 observation window.

**Time window:** March 2026.

The warehouse contains daily observations for each content page and client. For this lane, I aggregate those daily observations into one page/client record for March so the resulting features can support page prioritisation.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**

I use five March 2026 signals that are available at the time pages are prioritised:

- `march_gsc_impressions` — total GSC impressions during March. Available at the decision moment because it uses March search activity.
- `march_gsc_clicks` — total GSC clicks during March. Available at the decision moment because it uses March search activity.
- `march_gsc_ctr` — March clicks divided by March impressions. Available at the decision moment because it is calculated only from March search data.
- `march_ga4_sessions` — total GA4 sessions during March. Available at the decision moment when GA4 data is available.
- `march_scroll_events` — total scroll events during March. Available at the decision moment when GA4 data is available.

**Label / proxy**

The opportunity proxy flags a page when it had at least one GSC impression during March 2026 but received zero GSC clicks. This is a simple decision-support proxy for potential SEO opportunity, not proof that a page needs refreshing or that a refresh would improve performance.

**Context**

- `client_hash_id` — identifies the client associated with the page.
- `content_hash_id` — identifies the content page.
- `report_date` is used to define the March observation window and aggregate daily warehouse rows, but it is not retained as a page-level feature.

**Excluded**

I exclude April 2026 performance information from the decision-time features because it occurs after the March decision moment. Using future information would create label leakage and make the prioritisation score artificially optimistic.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is not set."

print("HF token loaded successfully.")

HF token loaded successfully.


In [23]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_file)

print("March 2026 rows:", len(march_df))

March 2026 rows: 9841378


Query 1: Verify the grain

In [24]:
grain_counts = (
    march_df
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="row_count")
)

print("Total rows:", len(march_df))
print("Unique date/client/content combinations:", len(grain_counts))
print("Maximum rows per combination:", grain_counts["row_count"].max())
print("Duplicate combinations:", (grain_counts["row_count"] > 1).sum())

Total rows: 9841378
Unique date/client/content combinations: 9841378
Maximum rows per combination: 1
Duplicate combinations: 0


Query 2: Verify row count and date span for March 2026

In [25]:
print("Row count:", len(march_df))
print("Minimum report date:", march_df["report_date"].min())
print("Maximum report date:", march_df["report_date"].max())
print("Unique report dates:", march_df["report_date"].nunique())

Row count: 9841378
Minimum report date: 2026-03-01
Maximum report date: 2026-03-31
Unique report dates: 31


Query 3: Verify GSC data availability

In [26]:
import duckdb

availability_result = duckdb.query("""
    SELECT
        COUNT(*) AS rows_before_filter,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*),
            1
        ) AS pct_surviving
    FROM march_df
""").df()

display(availability_result)

,rows_before_filter,rows_with_gsc,pct_surviving
0,9841378,3611061,36.7


### Five-feature frame

The following five features are aggregated from March 2026 daily observations to the page/client level. They use only information from the March decision window. GA4-based features are available only where GA4 data is available.

In [27]:
# Build March page-level feature frame

page_features = (
    march_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_gsc_impressions=("gsc_impressions", "sum"),
        march_gsc_clicks=("gsc_clicks", "sum"),
        march_ga4_sessions=("ga4_sessions", "sum"),
        march_scroll_events=("scroll_events", "sum"),
    )
)

# Calculate CTR from aggregated March totals
page_features["march_gsc_ctr"] = (
    page_features["march_gsc_clicks"]
    / page_features["march_gsc_impressions"].replace(0, pd.NA)
)

# Keep exactly five features
feature_cols = [
    "march_gsc_impressions",
    "march_gsc_clicks",
    "march_gsc_ctr",
    "march_ga4_sessions",
    "march_scroll_events",
]

feature_frame = page_features[
    ["client_hash_id", "content_hash_id"] + feature_cols
].copy()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

Feature frame shape: (331437, 7)


,client_hash_id,content_hash_id,march_gsc_impressions,march_gsc_clicks,march_gsc_ctr,march_ga4_sessions,march_scroll_events
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,<NA>,0.0,0.0
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,<NA>,0.0,0.0
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,<NA>,0.0,0.0
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,0.0,0.0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,<NA>,0.0,0.0


In [28]:
# Define a simple content opportunity proxy
# Opportunity = page had search impressions but received zero clicks

feature_frame["opportunity_proxy"] = (
    (feature_frame["march_gsc_impressions"] > 0)
    & (feature_frame["march_gsc_clicks"] == 0)
).astype(int)

print("Opportunity proxy distribution:")
print(feature_frame["opportunity_proxy"].value_counts())

print("\nOpportunity rate:")
print(f"{feature_frame['opportunity_proxy'].mean():.1%}")

Opportunity proxy distribution:
opportunity_proxy
0    223536
1    107901
Name: count, dtype: int64

Opportunity rate:
32.6%


### Deliberate leakage experiment

I intentionally introduce one label-derived column to demonstrate how leakage can make a scoring result look artificially strong. I then remove the leaked column and retain the honest score.

In [29]:
# Deliberate leakage experiment
# We intentionally use the proxy itself as a scoring feature.

K = 1000

# LEAKED score: directly uses the label/proxy
feature_frame["leaked_score"] = feature_frame["opportunity_proxy"]

top_k_leaked = feature_frame.nlargest(K, "leaked_score")

leaked_precision_at_k = top_k_leaked["opportunity_proxy"].mean()

print(f"Leaked Precision@{K}: {leaked_precision_at_k:.3f}")

Leaked Precision@1000: 1.000


In [30]:
# Remove the deliberately leaked label-derived column
feature_frame = feature_frame.drop(columns=["leaked_score"])

# Honest score using only decision-time features
# Higher impressions with fewer clicks indicates greater opportunity.
feature_frame["honest_score"] = (
    feature_frame["march_gsc_impressions"]
    / (feature_frame["march_gsc_clicks"] + 1)
)

top_k_honest = feature_frame.nlargest(K, "honest_score")

honest_precision_at_k = top_k_honest["opportunity_proxy"].mean()

print(f"Honest Precision@{K}: {honest_precision_at_k:.3f}")

Honest Precision@1000: 0.572


I deliberately created a label-derived `leaked_score` using the opportunity proxy itself. This produced a **Precision@1000 of 1.000**, demonstrating how using information derived from the target can create an artificially perfect result.

I then removed `leaked_score` and calculated a simple decision-time baseline using only March search signals. The resulting **Precision@1000 was 0.572**.

The difference shows why label-derived information must not be available to the scoring process at the decision moment. The honest score is the number I retain for comparison, while the leaked score is kept only as a demonstration of the leakage problem.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

GSC and GA4 data availability is uneven across the warehouse, so some pages have incomplete search or engagement signals. In particular, GA4 data is unavailable for many observations, so missing GA4 values cannot automatically be interpreted as zero activity.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.